# Day 1 · Exercise 1: How many shoes should the factory make?

**What you'll be able to say by the end**
1. "Stitching is the bottleneck, and one extra stitching hour is worth $27.50."
2. "Extra cutting or assembly hours are worth $0 right now, because those machines already have spare time."
3. "That $27.50 only holds near today's capacity. Past about 970 hours, a different machine becomes the bottleneck."

**How this notebook works:** first we solve the problem by hand with simple arithmetic. Then we let a solver do it and check that it agrees. Then we find a case where the hand method breaks, which is why solvers exist.

_Setup check: make sure this notebook runs on the repo's `.venv`. Nothing to learn here._

In [1]:
import sys
from pathlib import Path

VENV = (Path.cwd().parent / ".venv").resolve()
assert Path(sys.prefix).resolve() == VENV, f"Wrong kernel: {sys.prefix}. Select 'supply-chain-or (.venv)'."
print("Kernel:", sys.executable)

Kernel: /Users/abhijeet/Personal/supply_chain_or/.venv/bin/python


## 1. The story

A factory makes three kinds of shoes. Every pair needs some time on three machines: **cutting**, **stitching** and **assembly**. Each machine is only available for so many hours a week, and the market will only buy so many of each shoe.

**The question:** how many pairs of each shoe should we make this week to earn the most profit?

Here is all the data:

In [2]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

PRODUCTS = ["running", "basketball", "lifestyle"]
MACHINES = ["cutting", "stitching", "assembly"]

profit = {"running": 30, "basketball": 45, "lifestyle": 22}         # $ per pair
max_sales = {"running": 400, "basketball": 250, "lifestyle": 500}   # pairs we can sell
hours_available = {"cutting": 600, "stitching": 900, "assembly": 700}  # hours per week
hours_per_pair = {
    "cutting":   {"running": 0.6, "basketball": 0.9, "lifestyle": 0.4},
    "stitching": {"running": 1.0, "basketball": 1.5, "lifestyle": 0.8},
    "assembly":  {"running": 0.7, "basketball": 1.2, "lifestyle": 0.5},
}

table = pd.DataFrame(hours_per_pair).T
table.columns.name = "hours per pair"
table["hours available"] = pd.Series(hours_available)
table.loc["profit per pair ($)"] = [profit[p] for p in PRODUCTS] + [None]
table.loc["most we can sell"] = [max_sales[p] for p in PRODUCTS] + [None]
table

hours per pair,running,basketball,lifestyle,hours available
cutting,0.6,0.9,0.4,600.0
stitching,1.0,1.5,0.8,900.0
assembly,0.7,1.2,0.5,700.0
profit per pair ($),30.0,45.0,22.0,NaN
most we can sell,400.0,250.0,500.0,NaN


## 2. Why not just make everything we can sell?

Let's check how many hours that would take on each machine.

In [3]:
need = {m: sum(hours_per_pair[m][p] * max_sales[p] for p in PRODUCTS) for m in MACHINES}
check = pd.DataFrame({"hours needed": need, "hours available": hours_available})
check["short by"] = (check["hours needed"] - check["hours available"]).clip(lower=0)
check

,hours needed,hours available,short by
cutting,665.0,600,65.0
stitching,1175.0,900,275.0
assembly,830.0,700,130.0


**Read it like this:** making everything would need **1,175 stitching hours**, but we only have **900**. Cutting and assembly are short too, but stitching is short by the most.

So we can't make everything, and we have to choose. The machine that runs out first is called the **bottleneck**. Here it's stitching.

## 3. Solve it by hand

**The idea:** stitching hours are the scarce thing, so spend each one on whatever earns the most money per stitching hour.

Profit per stitching hour = profit per pair ÷ stitching hours per pair.

In [4]:
per_hour = pd.Series({p: profit[p] / hours_per_pair["stitching"][p] for p in PRODUCTS},
                     name="$ per stitching hour")
per_hour.sort_values(ascending=False)

running       30.0
basketball    30.0
lifestyle     27.5
Name: $ per stitching hour, dtype: float64

Running and basketball both earn **$30 per stitching hour**, and lifestyle earns **$27.50**. So we fill demand for running and basketball first, then spend whatever stitching time is left on lifestyle.

In [5]:
left = hours_available["stitching"]
hand_plan = {}
for p in per_hour.sort_values(ascending=False).index:
    can_make = left / hours_per_pair["stitching"][p]
    hand_plan[p] = min(max_sales[p], can_make)
    left -= hand_plan[p] * hours_per_pair["stitching"][p]
    print(f"{p:<11} make {hand_plan[p]:7.2f} pairs  -> stitching hours left: {left:6.1f}")

hand_profit = sum(profit[p] * hand_plan[p] for p in PRODUCTS)
print(f"\nProfit by hand: ${hand_profit:,.2f}")

running     make  400.00 pairs  -> stitching hours left:  500.0
basketball  make  250.00 pairs  -> stitching hours left:  125.0
lifestyle   make  156.25 pairs  -> stitching hours left:    0.0

Profit by hand: $26,687.50


**The plan:** 400 running + 250 basketball use 775 stitching hours. The remaining 125 hours make 125 ÷ 0.8 = 156.25 lifestyle pairs. Profit = **$26,687.50**.

(The 0.25 pair is fine for now. Day 2 covers forcing whole numbers.)

## 4. Now let a solver do it

A **linear program (LP)** is just this kind of problem written in a form a computer can solve. You describe three things:

| Piece | Plain words | In our problem |
|---|---|---|
| **Decision variables** | The numbers you get to choose | How many pairs of each shoe: `x_running`, `x_basketball`, `x_lifestyle` |
| **Objective** | What you want to make as big (or small) as possible | Total profit = 30·x_running + 45·x_basketball + 22·x_lifestyle |
| **Constraints** | The rules you can't break | Each machine's hours used ≤ hours available; each shoe made ≤ what we can sell; no negative pairs |

It's "linear" because everything is just numbers times variables, added up: no squares, no if-statements.

We'll use **PuLP**, a Python library that lets you write these three pieces almost exactly as in the table. It then hands them to a solver called **HiGHS**, which finds the best answer.

In [6]:
import pulp

def solve(hours_available):
    model = pulp.LpProblem("shoe_factory", pulp.LpMaximize)

    # Decision variables: pairs of each shoe (can't be negative)
    x = {p: model.add_variable(f"x_{p}", lowBound=0) for p in PRODUCTS}

    # Objective: total profit
    model += pulp.lpSum(profit[p] * x[p] for p in PRODUCTS)

    # Constraint 1: each machine's hours used <= hours available
    for m in MACHINES:
        model += pulp.lpSum(hours_per_pair[m][p] * x[p] for p in PRODUCTS) <= hours_available[m], f"hours_{m}"

    # Constraint 2: don't make more than we can sell
    for p in PRODUCTS:
        model += x[p] <= max_sales[p], f"sales_{p}"

    stats = model.solve(pulp.HiGHS(msg=False))
    assert stats.status == pulp.LpSolveStatus.Optimal, stats.status
    return model, x


model, x = solve(hours_available)
solver_profit = pulp.value(model.objective)

pd.DataFrame({
    "by hand": hand_plan,
    "solver": {p: x[p].varValue for p in PRODUCTS},
}).assign(**{"same?": lambda d: (d["by hand"] - d["solver"]).abs() < 1e-6})

,by hand,solver,same?
running,400.00,400.00,True
basketball,250.00,250.00,True
lifestyle,156.25,156.25,True


In [7]:
print(f"Profit by hand: ${hand_profit:,.2f}")
print(f"Profit solver:  ${solver_profit:,.2f}")

Profit by hand: $26,687.50
Profit solver:  $26,687.50


The solver found the same plan as our hand calculation. It isn't doing magic. It does the same reasoning, but it can handle thousands of products and limits at once.

## 5. The solver's bonus answers: slack and shadow prices

Besides the plan, the solver tells you two useful things about every constraint:

- **Slack:** how much of that limit is left over. Slack = 0 means the limit is fully used up, which is called **binding**.
- **Shadow price** (also called the **dual**): how much profit would rise if that limit were 1 unit bigger.

Here they are, with the reason for each in plain words:

In [8]:
why = {
    "hours_cutting":   "Has spare hours, so one more hour adds nothing",
    "hours_stitching": "Bottleneck. 1 more hour = 1/0.8 = 1.25 more lifestyle pairs x $22 = $27.50",
    "hours_assembly":  "Has spare hours, so one more hour adds nothing",
    "sales_running":   "1 more running pair: +$30, but uses 1 stitching hour taken from lifestyle (-$27.50) = +$2.50",
    "sales_basketball":"1 more basketball pair: +$45, but uses 1.5 stitching hours taken from lifestyle (-$41.25) = +$3.75",
    "sales_lifestyle": "We aren't selling all we could anyway, so more demand adds nothing",
}
pd.DataFrame(
    [{"constraint": c.name, "left over (slack)": round(c.slack, 2) + 0.0,
      "shadow price ($)": round(c.pi, 2), "why": why[c.name]} for c in model.constraints()]
).set_index("constraint")

,left over (slack),shadow price ($),why
constraint,,,
hours_cutting,72.50,0.00,"Has spare hours, so one more hour adds nothing"
hours_stitching,0.00,27.50,Bottleneck. 1 more hour = 1/0.8 = 1.25 more lifestyle pairs x $22 = $27.50
hours_assembly,41.88,0.00,"Has spare hours, so one more hour adds nothing"
sales_running,0.00,2.50,"1 more running pair: +$30, but uses 1 stitching hour taken from lifestyle (-$27.50) = +$2.50"
sales_basketball,0.00,3.75,"1 more basketball pair: +$45, but uses 1.5 stitching hours taken from lifestyle (-$41.25) = +$3.75"
sales_lifestyle,343.75,0.00,"We aren't selling all we could anyway, so more demand adds nothing"


**The pattern to remember:** if a limit has leftover (slack > 0), its shadow price is $0. Only the limits that are used up are worth paying to expand.

**What a manager does with this:** pay up to **$27.50/hour** for stitching overtime. Don't pay anything for more cutting or assembly time, because adding it changes nothing.

## 6. Prove the shadow prices: add 1 hour and re-solve

Don't take the shadow prices on trust. Give each machine one extra hour, solve again, and see how much profit actually went up.

In [9]:
rows = []
for m in MACHINES:
    plus_one = dict(hours_available, **{m: hours_available[m] + 1})
    model_plus, _ = solve(plus_one)
    rows.append({
        "machine": m,
        "shadow price said": model.get_constraint_by_name(f"hours_{m}").pi,
        "profit actually went up by": pulp.value(model_plus.objective) - solver_profit,
    })
pd.DataFrame(rows).set_index("machine").round(2)

,shadow price said,profit actually went up by
machine,,
cutting,0.0,0.0
stitching,27.5,27.5
assembly,0.0,0.0


They match exactly. A shadow price is just a fast way to get the answer to "what if I had one more unit of this?" without re-solving.

## 7. What if we had a lot more stitching time?

A shadow price is only true **near where you are today**. Let's vary stitching hours from 600 to 1,300 and watch what an extra hour is worth.

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

hours = list(range(600, 1301, 5))
profits, values = [], []
for h in hours:
    m_h, _ = solve(dict(hours_available, stitching=h))
    profits.append(pulp.value(m_h.objective))
    values.append(m_h.get_constraint_by_name("hours_stitching").pi)

# Where the value of an extra hour changes
breaks = [hours[i] for i in range(1, len(hours)) if abs(values[i] - values[i - 1]) > 1e-6]

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Total profit vs stitching hours", "What one extra stitching hour is worth"))
fig.add_trace(go.Scatter(x=hours, y=profits, mode="lines",
                         hovertemplate="%{x} h<br>profit $%{y:,.0f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=hours, y=values, mode="lines", line_shape="hv",
                         hovertemplate="%{x} h<br>$%{y:.2f} per extra hour<extra></extra>"), row=1, col=2)
for col in (1, 2):
    fig.add_vline(x=hours_available["stitching"], line_dash="dot", line_color="gray", row=1, col=col)
    for b in breaks:
        fig.add_vline(x=b, line_width=1, line_color="lightgray", row=1, col=col)
fig.add_annotation(x=hours_available["stitching"], y=max(profits), text="today: 900 h",
                   showarrow=False, yshift=10, row=1, col=1)
fig.update_xaxes(title_text="Stitching hours per week")
fig.update_yaxes(title_text="Profit", tickprefix="$", row=1, col=1)
fig.update_yaxes(title_text="$ per extra hour", tickprefix="$", row=1, col=2)
fig.update_layout(height=430, width=1000, showlegend=False, template="plotly_white")
fig.show()
print("Value of an extra hour changes near:", breaks, "hours")

Value of an extra hour changes near: [775, 970, 1015] hours


**Read the right-hand chart left to right:**

| Stitching hours | Extra hour worth | Why |
|---|---|---|
| 600 to 775 | **$30** | Not even enough to make all the running and basketball shoes, so each extra hour goes to them ($30/hour) |
| 775 to about 967 | **$27.50** | Running and basketball are maxed out, so extra hours go to lifestyle ($27.50/hour). **Today's 900 hours is here.** |
| about 967 to 1,012 | **$18.57** | **Assembly** runs out of hours too. Two machines are now tight, so each extra stitching hour is worth less |
| above about 1,012 | **$0** | Stitching isn't the bottleneck any more. Assembly is, so more stitching time changes nothing |

Each place where the line steps down is where a different limit becomes the bottleneck. The left-hand chart shows the same thing: profit climbs more slowly after each step, then goes flat.

## 8. Where the hand method breaks, and why we need solvers

Our hand rule was "spend stitching hours on the best $ per stitching hour". That works when there's **one** bottleneck. Give the factory 1,100 stitching hours and assembly becomes the bottleneck instead. Let's see what the hand rule does then.

In [11]:
big = dict(hours_available, stitching=1100)

# Hand rule: same ranking by $ per stitching hour, but now also respect the other machines
left = dict(big)
hand_plan_2 = {}
for p in per_hour.sort_values(ascending=False).index:
    can_make = min(left[m] / hours_per_pair[m][p] for m in MACHINES)
    hand_plan_2[p] = min(max_sales[p], can_make)
    for m in MACHINES:
        left[m] -= hand_plan_2[p] * hours_per_pair[m][p]

model_2, x_2 = solve(big)
compare = pd.DataFrame({"hand rule": hand_plan_2, "solver": {p: x_2[p].varValue for p in PRODUCTS}}).round(1)
compare.loc["profit ($)"] = [sum(profit[p] * hand_plan_2[p] for p in PRODUCTS), pulp.value(model_2.objective)]
compare

,hand rule,solver
running,400.0,400.0
basketball,250.0,141.7
lifestyle,240.0,500.0
profit ($),28530.0,29375.0


The solver earns **$845 more** than the hand rule. How?

Now **assembly** is the scarce machine, so the right question is "$ per **assembly** hour":
- Lifestyle: $22 ÷ 0.5 h = **$44.00** per assembly hour
- Running: $30 ÷ 0.7 h = **$42.86** per assembly hour
- Basketball: $45 ÷ 1.2 h = **$37.50** per assembly hour

So the solver **makes fewer basketball shoes**, even though they have the highest profit per pair, and uses that assembly time for lifestyle instead.

With 3 products and 3 machines you could still work this out by hand. Nike has thousands of products, dozens of factories and DCs, and many weeks to plan, with bottlenecks shifting all the time. That's the job of an LP solver.

## Recap: say these out loud

1. **What's an LP?** You choose some numbers (decision variables) to maximize or minimize a goal (objective), without breaking some rules (constraints). Everything is numbers times variables, added up.
2. **What's the answer here?** 400 running, 250 basketball, 156 lifestyle, for $26,687.50 profit.
3. **What's the bottleneck?** Stitching. It's fully used (slack = 0); cutting and assembly have spare hours.
4. **What's an extra stitching hour worth?** $27.50, its shadow price. Extra cutting or assembly hours are worth $0.
5. **Is that always true?** No. It holds from about 775 to 967 hours. Past that, assembly becomes the bottleneck too and the value drops.
6. **Why not just use a rule of thumb?** With several bottlenecks at once, simple rules miss money: $845 here, much more at Nike's scale.

**Words from today**

| Word | Meaning |
|---|---|
| Decision variable | A number you choose (pairs to make) |
| Objective | The goal you maximize or minimize (profit) |
| Constraint | A rule you can't break (machine hours, sales limit) |
| Bottleneck | The limit that runs out first |
| Binding | A constraint that's fully used up |
| Slack | How much of a limit is left over |
| Shadow price / dual | Profit gained from 1 more unit of a limit. $0 when there's slack |

**Try it yourself:** change `profit["lifestyle"]` to 30 and re-run everything. Before you do, predict: which shoe is now best per stitching hour, and what happens to the shadow price?